# Notebook 07: Kế hoạch Can thiệp Cá nhân (Counterfactual Explanations) - Phase 3

**Dự án:** EduGuard: AI-Powered Early Warning and Actionable Recourse System for Students
**Nhóm thực hiện:** Nhóm 5, DSP391m

---
## Mục tiêu của Notebook

Đây là **Giai đoạn 3 (Phase 3)** của hệ thống EduGuard. Sau khi mô hình đã dự đoán sinh viên có nguy cơ (Phase 1) và giải thích lý do bằng SHAP (Phase 2), giai đoạn này đóng vai trò là bước **"kê đơn thuốc"**.

Thay vì chỉ báo động, chúng ta sử dụng phương pháp **Counterfactual Explanations** (Giải thích bằng Phản thực tế) để tìm ra những hành động tối thiểu mà một sinh viên có nguy cơ cần thực hiện (ví dụ: tăng số lượt truy cập, nộp thêm bài) để vượt qua ngưỡng an toàn.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.evaluation.make_split import load_checkpoint_split
from src.features.preprocessing import make_X_y

import warnings
warnings.filterwarnings('ignore')

## 1. Thiết lập các đặc trưng có thể thay đổi (Actionable Features)

Một nguyên tắc đạo đức quan trọng trong AI giáo dục: **Không yêu cầu sinh viên thay đổi những thứ không thể thay đổi**. 

Các đặc trưng nhân khẩu học (Tuổi, Giới tính, Khu vực, Khuyết tật) là **bất biến (Immutable)**. Chúng ta chỉ tạo ra kế hoạch can thiệp trên các đặc trưng **hành vi học tập (Actionable)**.

In [ ]:
ACTIONABLE_FEATURES = {
    "total_clicks": "Tăng",
    "n_days_active": "Tăng",
    "mean_clicks_per_active_day": "Tăng",
    "clicks_forumng": "Tăng",
    "clicks_oucontent": "Tăng",
    "clicks_resource": "Tăng",
    "n_assessments_submitted": "Tăng",
    "weighted_score_to_date": "Tăng",
    "not_submitted": "Giảm",
    "days_since_last_activity": "Giảm",
}

print("Các hành vi có thể thay đổi để can thiệp:")
for feat, direction in ACTIONABLE_FEATURES.items():
    print(f" - {feat}: Hướng mục tiêu -> {direction}")

## 2. Tìm kiếm nhóm Sinh viên An toàn (Safe Cohort)

Trong thực tế triển khai ở app EduGuard, để đưa ra gợi ý khả thi nhất (Phương pháp Khoảng cách nhỏ nhất - Minimum Distance), chúng ta lấy **Median (Trung vị)** của những sinh viên an toàn (những người không có nguy cơ trượt) làm "Mục tiêu" (Target) cho sinh viên có nguy cơ phấn đấu theo.

In [ ]:
t_checkpoint = 50 # Giả lập mốc 50% khóa học

train_df, test_df = load_checkpoint_split(t_checkpoint)
X_train, y_train = make_X_y(train_df)
X_test, y_test = make_X_y(test_df)

# 0 = Pass/Distinction (An toàn), 1 = Fail/Withdrawn (Nguy cơ)
safe_students = X_train[y_train == 0]
at_risk_students = X_train[y_train == 1]

print(f"Số lượng SV an toàn tại mốc {t_checkpoint}%: {len(safe_students)}")
print(f"Số lượng SV có nguy cơ tại mốc {t_checkpoint}%: {len(at_risk_students)}")

# Tính toán Median cho nhóm an toàn (Mục tiêu lý tưởng)
safe_medians = safe_students[list(ACTIONABLE_FEATURES.keys())].median()
display(pd.DataFrame(safe_medians, columns=['Mục tiêu (Median của SV An toàn)']))

## 3. Sinh Kế hoạch Can thiệp Đa dạng (Diverse Counterfactuals)

Chúng ta sẽ chọn 1 sinh viên có nguy cơ và đưa ra 3 phương án khác nhau để họ lựa chọn.

In [ ]:
# Chọn ngẫu nhiên 1 sinh viên có nguy cơ
student_id = at_risk_students.index[0]
student_profile = at_risk_students.loc[student_id]

def generate_counterfactual_plan(student, medians, feats_up, feats_down, factor_up=1.0, factor_down=0.5):
    plan = {}
    for feat in feats_up:
        current_val = student[feat]
        target_val = medians[feat]
        if target_val > current_val:
            plan[feat] = {'Current': current_val, 'Target': target_val, 'Change': target_val - current_val}
            
    for feat in feats_down:
        current_val = student[feat]
        target_val = medians[feat] * factor_down
        if target_val < current_val:
            plan[feat] = {'Current': current_val, 'Target': max(0, target_val), 'Change': target_val - current_val}
            
    return pd.DataFrame(plan).T

# Phương án A: Tăng cường hoạt động VLE
plan_A = generate_counterfactual_plan(
    student_profile, safe_medians, 
    feats_up=['total_clicks', 'n_days_active', 'clicks_oucontent', 'clicks_forumng'],
    feats_down=['days_since_last_activity']
)

# Phương án B: Tập trung làm bài tập
plan_B = generate_counterfactual_plan(
    student_profile, safe_medians,
    feats_up=['n_assessments_submitted', 'weighted_score_to_date'],
    feats_down=['not_submitted']
)

print(f"== KẾ HOẠCH CAN THIỆP CHO SINH VIÊN {student_id} ==\n")
print("PHƯƠNG ÁN A: Tăng cường tham gia VLE (Diễn đàn, Bài giảng)")
display(plan_A)

print("\nPHƯƠNG ÁN B: Tập trung hoàn thành bài tập (Điểm số)")
display(plan_B)


## 4. Kết luận Giai đoạn 3
- Bằng cách cung cấp các kế hoạch hành động cụ thể, **EduGuard vượt ra khỏi khuôn khổ của một hệ thống cảnh báo (Warning System)** thông thường, trở thành một **hệ thống gợi ý can thiệp (Actionable Recourse System)**.
- Điều này giúp sinh viên và giảng viên biết chính xác phải làm gì tiếp theo thay vì hoang mang trước một con số xác suất nguy cơ.